เซลล์ที่ 1: ติดตั้งและเชื่อมต่อ Neo4j

In [ ]:
!pip install -q neo4j
import neo4j
print("Neo4j Python Driver Version:", neo4j.__version__)

from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://4cd28f11.databases.neo4j.io"
USERNAME = "4cd28f11"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()
print("Connected successfully")

result = driver.execute_query("SHOW HOME DATABASE")
DATABASE = result.records[0]["name"]
print("Database =", DATABASE)



Neo4j Python Driver Version: 6.3.1
Password: ··········
Connected successfully
Database = 4cd28f11


In [ ]:
driver.execute_query(
    """
    MATCH (n)
    WHERE n:User OR n:Drink OR n:Category
    DETACH DELETE n
    """,
    database_=DATABASE
)

print("ล้างข้อมูลเดิมเรียบร้อย")

ล้างข้อมูลเดิมเรียบร้อย


สร้าง Constraint

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique IF NOT EXISTS
    FOR (u:User) REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

driver.execute_query(
    """
    CREATE CONSTRAINT drink_name_unique IF NOT EXISTS
    FOR (d:Drink) REQUIRE d.name IS UNIQUE
    """,
    database_=DATABASE
)

print("สร้าง Constraint สำเร็จ")

สร้าง Constraint สำเร็จ


สร้าง Node ผู้ใช้, เครื่องดื่ม และความสัมพันธ์ (DRINKS & FRIEND)

In [ ]:
users = [
    "Anan", "Bee", "Chai", "Dao", "Earn",
    "Fah", "Gun", "Ice", "Jane", "Kong"
]

drinks = [
    "Latte", "Mocha", "Espresso", "Americano",
    "Matcha", "Cocoa", "ThaiTea", "Smoothie"
]

# สร้าง User และ Drink Nodes
driver.execute_query(
    """
    UNWIND $users AS name MERGE (u:User {name: name})
    """,
    users=users, database_=DATABASE
)

driver.execute_query(
    """
    UNWIND $drinks AS name MERGE (d:Drink {name: name})
    """,
    drinks=drinks, database_=DATABASE
)

# 1. ความสัมพันธ์การดื่มน้ำ (DRINKS)
drinks_rel = [
    ("Anan", "Latte"), ("Anan", "Mocha"),
    ("Bee", "Latte"), ("Bee", "Mocha"), ("Bee", "Americano"),
    ("Chai", "Latte"), ("Chai", "Espresso"),
    ("Dao", "Americano"), ("Dao", "ThaiTea"),
    ("Earn", "Mocha"), ("Earn", "Americano"), ("Earn", "Matcha"),
    ("Fah", "Latte"), ("Fah", "Cocoa"),
    ("Gun", "ThaiTea"), ("Gun", "Smoothie"),
    ("Ice", "Mocha"), ("Ice", "Matcha"), ("Ice", "Cocoa"),
    ("Jane", "Latte"), ("Jane", "Mocha"), ("Jane", "Espresso"), ("Jane", "Americano"),
    ("Kong", "Americano"), ("Kong", "Smoothie"), ("Kong", "ThaiTea")
]

driver.execute_query(
    """
    UNWIND $rows AS row
    MATCH (u:User {name: row.user}), (d:Drink {name: row.drink})
    MERGE (u)-[:DRINKS]->(d)
    """,
    rows=[{"user": u, "drink": d} for u, d in drinks_rel],
    database_=DATABASE
)

# 2. ความสัมพันธ์การเป็นเพื่อนกัน (FRIEND - แบบสองทิศทางหรือสมมติกลุ่มเพื่อน)
friendships = [
    ("Anan", "Bee"), ("Anan", "Chai"), ("Anan", "Earn"),
    ("Bee", "Jane"), ("Bee", "Earn"),
    ("Chai", "Jane"), ("Dao", "Kong"), ("Dao", "Gun"),
    ("Earn", "Jane"), ("Fah", "Anan"), ("Gun", "Kong"),
    ("Ice", "Earn"), ("Jane", "Anan"), ("Kong", "Dao")
]

driver.execute_query(
    """
    UNWIND $rows AS row
    MATCH (u1:User {name: row.u1}), (u2:User {name: row.u2})
    MERGE (u1)-[:FRIEND]-(u2)
    """,
    rows=[{"u1": f[0], "u2": f[1]} for f in friendships],
    database_=DATABASE
)

print("สร้างข้อมูลผู้ใช้, เครื่องดื่ม, DRINKS และ FRIEND สำเร็จ")

สร้างข้อมูลผู้ใช้, เครื่องดื่ม, DRINKS และ FRIEND สำเร็จ


ฟังก์ชันแนะนำเครื่องดื่มแบบ Friend-of-Friend (FoF)

In [ ]:
def recommend_by_friend(target_user, top_n=None):
    """
    แนวคิด: ตัวเรา -> เพื่อน -> เครื่องดื่มที่เพื่อนชอบ (แต่เรายังไม่เคยดื่ม)
    """
    query = """
    MATCH
        (me:User {name: $name})
        -[:FRIEND]-
        (friend:User)
        -[:DRINKS]->
        (rec:Drink)
    WHERE NOT EXISTS {
          MATCH (me)-[:DRINKS]->(rec)
      }
      AND friend <> me
    RETURN
        rec.name AS drink,
        collect(DISTINCT friend.name) AS recommended_by_friends,
        count(DISTINCT friend) AS score
    ORDER BY
        score DESC,
        drink
    """

    if top_n is not None:
        query += "\nLIMIT $top_n"

    result = driver.execute_query(
        query,
        name=target_user,
        top_n=top_n,
        database_=DATABASE
    )

    return [
        {
            "drink": record["drink"],
            "score": record["score"],
            "friends": record["recommended_by_friends"]
        }
        for record in result.records
    ]

# ทดสอบรันดูผลลัพธ์ของ Anan
print(f"=== ผลการแนะนำเครื่องดื่มให้ Anan (ผ่านทางเพื่อน) ===")
for rec in recommend_by_friend("Anan"):
    print(f"เครื่องดื่ม: {rec['drink']} | คะแนน (จำนวนเพื่อนที่ดื่ม): {rec['score']} | แนะนำโดยเพื่อน: {rec['friends']}")

=== ผลการแนะนำเครื่องดื่มให้ Anan (ผ่านทางเพื่อน) ===
เครื่องดื่ม: Americano | คะแนน (จำนวนเพื่อนที่ดื่ม): 3 | แนะนำโดยเพื่อน: ['Bee', 'Earn', 'Jane']
เครื่องดื่ม: Espresso | คะแนน (จำนวนเพื่อนที่ดื่ม): 2 | แนะนำโดยเพื่อน: ['Chai', 'Jane']
เครื่องดื่ม: Cocoa | คะแนน (จำนวนเพื่อนที่ดื่ม): 1 | แนะนำโดยเพื่อน: ['Fah']
เครื่องดื่ม: Matcha | คะแนน (จำนวนเพื่อนที่ดื่ม): 1 | แนะนำโดยเพื่อน: ['Earn']
